# Building the viewer catalog

The viewer reads one `catalog.json` per example. This notebook builds one example a join at a time so each table can be inspected. `pixi run collect && pixi run build` runs the same functions end to end.

`data/<example>/` holds symlinks to the source products (made by `collect`); nothing at the source is modified.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import geopandas as gpd
import pandas as pd

import catalog
import collect
import gibs
import opera

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

## Configuration

Pick the example here. `sources` are where the products live; `data` is this repo's folder of links to them.

In [ ]:
EXAMPLE = 'validation'
cfg = catalog.example_config(EXAMPLE)
cfg['sources'], cfg['data']

In [ ]:
collect.collect_example(cfg['sources'], cfg['data'])

## 1. nisar-water run

Products are filed by run: `v<version>` for a tagged release, `dev-<git_sha>` otherwise. The newest run is used unless one is named.

In [ ]:
RUN = None
run_dir = catalog.pick_run(cfg['data'], RUN)
[p.name for p in catalog.run_dirs(cfg['data'])], run_dir

## 2. Scenes

One row per NISAR frame and date. `wtr` is the frame product for this run, or `None` if it hasn't been processed.

In [ ]:
scenes = catalog.read_scenes(cfg['data'], run_dir)
scenes[['scene', 'frame_id', 'direction', 'acq_key', 'wtr']].head()

## 3. DSWx-NI tiles

Parsed from file names. The acquisition start time links a tile to a scene; when two deliveries cover one acquisition, the newest processing time wins.

In [ ]:
tiles = catalog.dswx_ni_tiles(cfg['data'])
tiles.groupby('version').size(), tiles.head()

In [ ]:
jobs = catalog.dswx_ni_jobs(EXAMPLE, scenes, cfg['data'])
scene = next(iter(jobs))
sources, dst = jobs[scene]
scene, [s.name for s in sources], dst.relative_to(catalog.ROOT)

## 4. Validation sites (validation only)

UMD chips with Good NISAR GCOV, labelled with state/province and country from Natural Earth.

In [ ]:
if cfg['kind'] == 'sites':
    sites = catalog.validation_sites(cfg['data'] / 'tables')
    display(sites[['site_id', 'label', 'quality', 'comments', 'nisar_id']].head(10))
    display(sites.country.value_counts().head(10))

## 5. DSWx-HLS, DSWx-S1 and Sentinel-1 dates by coverage

Candidates are every date within ±7 days with a granule touching the site (public CMR search). Each candidate is scored by the share of the site polygon with valid data in the GIBS tiles the viewer shows (cloud counts as invalid for HLS). The nearest date with ≥ 90 % coverage wins, else the best-covered one.

In [ ]:
row = scenes.iloc[-1]
if cfg['kind'] == 'sites':
    footprint = sites.set_index('nisar_id').loc[row.granule_id].geometry
else:
    footprint = gpd.read_file(cfg['data'] / 'aoi.geojson').union_all()
candidates = {key: opera.date_candidates(short, catalog.bbox(footprint), row.acq_time) for key, short in opera.SHORT_NAMES.items()}
pd.DataFrame([{'layer': k, 'date': c['date'], 'days': c['days'], 'granules': len(c['granules']), 'coverage': gibs.coverage(k, c['date'], footprint.wkt)} for k, cs in candidates.items() for c in cs])

In [ ]:
opera.best_dates([(key, catalog.bbox(footprint), footprint.wkt, row.acq_time) for key in opera.SHORT_NAMES])

## 6. Build

Writes web-mercator COGs (skipping existing ones), fills in the remaining OPERA dates, and writes `build/<example>/catalog.json` and `sites.geojson`.

In [ ]:
built = catalog.build_example(EXAMPLE, RUN)
catalog.summary(built)

In [ ]:
pd.DataFrame(
    [{'scene': k, 'nisar_water': s['nisar_water'], 'dswx_ni': s['dswx_ni'], **{g: (v or {}).get('date') for g, v in s['gibs'].items()}} for k, s in built['scenes'].items()]
).head(10)